# 06 — PARTITIONS symboliques : la musique devient du texte

Pour « parler djembé », il faut d'abord écrire le djembé. On convertit chaque prise en **partition symbolique** : pour chaque subdivision du cycle (1–12), on note :
- `D`  : le dunun frappe (`x`) ou pas (`-`)
- `J2` : l'accompagnement frappe ou pas
- `J1` : le soliste joue une **b**asse, une **t**onique, un **s**lap, ou rien (`-`)

Ingrédients : la **grille métrique** annotée (`Onsets_Selected`) + notre **typage des frappes** (clustering physique, notebook 02) accéléré par une extraction FFT vectorisée (~20 s pour 32 000 frappes).

Sortie : `output/partitions.csv` — le « texte » sur lequel le modèle génératif (07) apprendra.

## 0. Typage rapide de TOUTES les frappes de jembé (avec cache)

In [ ]:
import os, warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import djembe_utils as dj

CACHE = 'output/all_jembe_strokes_typed.csv'
c = dj.list_corpus()
if os.path.exists(CACHE):
    strokes = pd.read_csv(CACHE)
    print('Cache chargé :', len(strokes), 'frappes typées')
else:
    parts = []
    for _, r in c[c['role'].str.startswith('jembe')].iterrows():
        y = dj.load_track(r['piece'], r['take'], r['code'])
        on = dj.load_onsets(r['piece'], r['take'], r['code'])
        f = dj.fast_stroke_features(y, on['Time'].values)
        f['piece'], f['take'], f['code'], f['role'] = r['piece'], r['take'], r['code'], r['role']
        parts.append(f)
    strokes = pd.concat(parts, ignore_index=True)
    X = StandardScaler().fit_transform(strokes[dj.FAST_FEATS])
    strokes['cl'] = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X)
    strokes['stroke'] = dj.label_stroke_clusters(strokes)
    strokes.to_csv(CACHE, index=False)
    print('Construit :', len(strokes), 'frappes typées (cache écrit)')
if 'stroke' not in strokes.columns:
    X = StandardScaler().fit_transform(strokes[dj.FAST_FEATS])
    strokes['cl'] = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X)
    strokes['stroke'] = dj.label_stroke_clusters(strokes)
display(strokes.groupby('stroke').size().rename('n'))

## 1. Construire la partition d'une prise
On parcourt la grille (`Onsets_Selected`) et, pour chaque frappe de J1, on retrouve son type (appariement au plus proche, tolérance 40 ms).

In [ ]:
SYM = {'bass':'b', 'tone':'t', 'slap':'s'}

def build_partition(piece, take):
    sel = dj.load_selected(piece, take)
    if sel is None:
        return None
    j1 = strokes[(strokes['piece']==piece) & (strokes['take']==take) &
                 (strokes['code']=='J1')].sort_values('time')
    tj, sj = j1['time'].values, j1['stroke'].values
    rows = []
    for _, r in sel.iterrows():
        D  = 'x' if pd.notna(r.get('D1')) or pd.notna(r.get('D2')) else '-'
        J2 = 'x' if pd.notna(r.get('J2')) else '-'
        J1 = '-'
        if pd.notna(r.get('J1')) and len(tj):
            i = int(np.argmin(np.abs(tj - r['J1'])))
            J1 = SYM[sj[i]] if abs(tj[i] - r['J1']) < 0.04 else '?'
        rows.append({'piece':piece, 'take':take, 'label':r['Label SD'],
                     'SD':int(r['SD']), 'D':D, 'J2':J2, 'J1':J1})
    return pd.DataFrame(rows)

demo = build_partition('Woloso', 1)
print('Woloso_1 — premier cycle :')
display(demo.head(12)[['SD','D','J2','J1']])

## 2. Toutes les prises → le corpus de « texte djembé »

In [ ]:
parts = []
for (piece, take), _ in c.groupby(['piece','take']):
    p = build_partition(piece, take)
    if p is not None:
        parts.append(p)
score = pd.concat(parts, ignore_index=True)
print(len(score), 'subdivisions transcrites,', score['J1'].ne('-').sum(), 'frappes de soliste')
print('taux J1 non apparié (?) :', round((score['J1']=='?').mean(), 4))
score.to_csv('output/partitions.csv', index=False)
print('Sauvegardé -> output/partitions.csv')

## 3. Le « langage » du soliste, pièce par pièce
Quel type de frappe le soliste place-t-il à chaque position du cycle ? C'est sa **grammaire**.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
COL = {'b':'#2b6cff', 't':'#d63bd6', 's':'#2ec45f'}
for a, piece in zip(ax, dj.PIECES):
    sub = score[score['piece']==piece]
    tab = pd.crosstab(sub['SD'], sub['J1'], normalize='index')
    bottom = np.zeros(12)
    for s in ['b','t','s']:
        if s in tab:
            a.bar(tab.index, tab[s], bottom=bottom, color=COL[s], label=s)
            bottom += tab[s].values
    a.set_title(piece); a.set_xlabel('subdivision'); a.set_xticks(range(1,13))
ax[0].set_ylabel('proba de frappe J1 (par type)'); ax[0].legend(title='frappe')
plt.suptitle('La grammaire du soliste : quel type de frappe, à quelle position'); plt.tight_layout(); plt.show()

## Bilan
- Chaque prise est maintenant un **texte** : séquence de symboles (D, J2, J1∈{-,b,t,s}) sur la grille des 12 subdivisions.
- Taux d'appariement quasi parfait (<1 % de `?`).
- → `07_generation` : apprendre ce langage (Markov) et **générer** du djembé audible.